# RAG 101 - Corpus + Answerability

> *Corpus:* IBM mt-rag-benchmark government-services passages (49k docs).

**Duration:** ~15 min (first run, includes corpus embedding)

The smallest end-to-end RAG demo this repo offers: build a vector corpus, retrieve passages for a query, and ask the model **"can these passages actually answer it?"**. No generation, no citations, no clarification - just the gate that decides whether RAG should even attempt an answer.

*Why vLLM:* much faster inference in production environments; HF support for Granite Switch in mellea coming.

**What you'll learn:**
- How to stand up a ChromaDB corpus from a real-world dataset (49k IBM mt-rag-benchmark government-services passages) and query it.
- How `rag.check_answerability` decides whether retrieved documents can support an answer - the foundation that the larger RAG pipelines build on.
- How to recognize the **unanswerable** exit, so your application can refuse instead of hallucinating.

**Adapters used:** the `answerability` intrinsic from the [RAG](https://huggingface.co/ibm-granite/granitelib-rag-r1.0) library.

---
**Prerequisites:** GPU runtime (T4 or better). Go to *Runtime → Change runtime type → T4 GPU*.

New to mellea intrinsics? Start with [`hello_mellea.ipynb`](./hello_mellea.ipynb) for a softer walkthrough of each intrinsic in isolation. Full setup details (GPU sizes, HF auth, multi-GPU) are in [`PREREQUISITES.md`](../PREREQUISITES.md).

## 0 · Install and set up

In [ ]:
# Install granite-switch with tutorial dependencies (includes vLLM backend).
%pip install -q "granite-switch[tutorials]"

In [ ]:
import os
from pathlib import Path

from huggingface_hub import notebook_login

from granite_switch.tutorials.govt_data_loader import load_or_build_govt_chroma
from granite_switch.tutorials.vllm_server import (
    kill_stale_vllm_processes,
    launch_vllm,
    print_gpu_state,
    tail_log,
    wait_for_server,
)
from mellea.backends.openai import OpenAIBackend
from mellea.stdlib.components import Document as MelleaDocument
from mellea.stdlib.components.intrinsic import rag
from mellea.stdlib.context import ChatContext

try:
    from dotenv import load_dotenv
    load_dotenv(Path("../.env"), override=False)
except ImportError:
    pass

In [ ]:
notebook_login()  # needed to pull ibm-granite models from the Hub

kill_stale_vllm_processes()
print_gpu_state()

## 1 · Launch vLLM server

Start the Granite Switch model on port 8000. The server runs in the background; `wait_for_server` polls `/health` until it's ready.

In [ ]:
VLLM_MODEL = "ibm-granite/granite-switch-4.1-3b-preview"
VLLM_PORT = 8000
MAX_MODEL_LEN = 10240  # 10k, fits comfortably on an T4 GPU.

vllm_proc = launch_vllm(
    model=VLLM_MODEL,
    port=VLLM_PORT,
    max_model_len=MAX_MODEL_LEN,
    log_file="/content/vllm_server.log",
)
if not wait_for_server(VLLM_PORT):
    tail_log("/content/vllm_server.log")

## 2 · Configuration
Endpoints, model IDs, and corpus paths. Every value falls back to a sensible default, so the cell runs as-is if your vLLM server is on `localhost:8000`.

In [ ]:
# ── vLLM server ───────────────────────────────────────────────────────────────
# URL of the running vLLM OpenAI-compatible endpoint.
VLLM_BASE_URL = os.environ.get("VLLM_BASE_URL", "http://localhost:8000/v1")

# Model name as reported by GET /v1/models (usually the path/repo used at launch).
VLLM_MODEL_NAME = os.environ.get("VLLM_MODEL_NAME", "ibm-granite/granite-switch-4.1-3b-preview")

# HF Hub repo ID (or local path) to load I/O configs for the embedded adapters.
GRANITE_SWITCH_SOURCE = os.environ.get("GRANITE_SWITCH_SOURCE", VLLM_MODEL_NAME)

# ── Embedding model + ChromaDB persistence ───────────────────────────────────
EMBEDDING_MODEL_ID = "ibm-granite/granite-embedding-small-english-r2"
CHROMA_PATH = "./govt_chroma"

# ── Corpus source (only fetched when building from scratch) ──────────────────
GOVT_JSONL_URL  = "https://github.com/IBM/mt-rag-benchmark/raw/main/corpora/passage_level/govt.jsonl.zip"
GOVT_JSONL_PATH = "./govt.jsonl"

# TOP_K = passages retrieved per query. Small here because we only feed them to
# answerability — no generation budget to worry about.
TOP_K = 5

print(f"vLLM:      {VLLM_BASE_URL}  ({VLLM_MODEL_NAME})")
print(f"Embedding: {EMBEDDING_MODEL_ID}")
print(f"ChromaDB:  {CHROMA_PATH}")

## 3 · Build or load the vector corpus

`load_or_build_govt_chroma` is the corpus half of RAG, packaged so this notebook stays focused on retrieval and answerability:

1. Downloads `govt.jsonl.zip` (~50 MB, 49k government-service passages from [IBM mt-rag-benchmark](https://github.com/IBM/mt-rag-benchmark)) on first run.
2. Embeds each passage with `ibm-granite/granite-embedding-small-english-r2`.
3. Persists the index to `./govt_chroma` so subsequent runs load instantly.

> **Note:** to keep the tutorial fast, we filter most non-related docs and embed only the curated subset that the demo queries actually retrieve. For a full corpus load, set `load_only_tutorial_docs=False` in the call below.

In [ ]:
chroma_collection = load_or_build_govt_chroma(
    chroma_path             = CHROMA_PATH,
    jsonl_path              = GOVT_JSONL_PATH,
    jsonl_url               = GOVT_JSONL_URL,
    embedding_model_id      = EMBEDDING_MODEL_ID,
    load_only_tutorial_docs = True,
    device                  = "cpu",
)
print(f"Corpus ready — {chroma_collection.count():,} passages indexed.")

## 4 · Connect to vLLM via mellea
Registers the Granite Switch embedded adapters so `rag.check_answerability` routes to the correct control token.

In [ ]:
backend = OpenAIBackend(
    model_id=VLLM_MODEL_NAME,
    base_url=VLLM_BASE_URL,
    api_key="unused",
)
backend.register_embedded_adapter_model(GRANITE_SWITCH_SOURCE)
print(f"Backend ready — adapters: {backend.list_adapters()}")

## 5 · Ask: can the corpus answer this?

Two steps per query:

1. **Retrieve** the top-K most similar passages from ChromaDB.
2. **Check answerability** — `rag.check_answerability` returns the string `"answerable"` or `"unanswerable"`.

That's the entire RAG gate. In a full pipeline you'd only call the generation model when the verdict is `answerable`; on `unanswerable` you refuse and tell the user the corpus doesn't cover their question.

In [ ]:
def answerable(query: str, top_k: int = TOP_K) -> str:
    """Retrieve top-K passages and ask the model whether they can answer the query."""
    results = chroma_collection.query(query_texts=[query], n_results=top_k)
    docs = [MelleaDocument(doc_id=str(i), text=t) for i, t in enumerate(results["documents"][0])]

    verdict = rag.check_answerability(query, docs, ChatContext(), backend)

    glyph = "✅" if verdict == "answerable" else "🔍"
    preview = " ".join(docs[0].text.split())
    print(f"{glyph} {verdict.upper():13s}  query: {query}")
    print(f"retrieved {len(docs)} passages; top match: {preview}")
    return verdict

In [ ]:
# Q1 — answerable: government-service questions are exactly what this corpus covers.
_ = answerable("How do I check the status of my federal tax refund?")

In [ ]:
# Q2 — unanswerable: cooking is out of scope for a government-services corpus,
# even though retrieval will still return its top-K most-similar passages.
_ = answerable("What's the best recipe for sourdough bread?")

## 6 · Next steps

- **Add the rest of the pipeline.** [`rag_full_pipeline.ipynb`](./rag_full_pipeline.ipynb) layers query rewrite, clarification, grounded generation, citations, and guardian harm + scope checks on top of the same corpus and answerability check.
- **Compose your own checkpoint.** [`compose_granite_switch.ipynb`](./compose_granite_switch.ipynb) walks through building a Granite Switch model from the IBM adapter libraries.
- **Watch ALORA vs LoRA race.** [`alora_vs_lora_race.ipynb`](./alora_vs_lora_race.ipynb) compares the two activation styles head-to-head on the same workload.